# novel-ocr : 소설 스캔 이미지 → 텍스트 추출

Colab에서 실행하는 용도의 노트북입니다. 순서대로 셀을 실행하세요 (Shift+Enter).
OCR 엔진은 **PaddleOCR**(한글 인쇄체 정확도가 EasyOCR보다 좋음)을 사용합니다.

**사용 순서**
1. 런타임을 GPU로 설정 (상단 메뉴: 런타임 > 런타임 유형 변경 > T4 GPU)
2. 아래 셀을 위에서부터 순서대로 실행
3. Google Drive에 스캔 이미지(또는 zip)를 올려두고 경로만 맞춰주면 됨
4. 결과 텍스트 파일들이 원본과 같은 폴더 구조로 Drive에 자동 저장됨

## 1. 라이브러리 설치

한글 인쇄체 인식 정확도가 EasyOCR보다 나은 **PaddleOCR**을 사용합니다.
(받침/유사 자모 구분에서 EasyOCR보다 오류가 적습니다.)

In [ ]:
# PaddleOCR 설치 (GPU 버전 시도).
# 만약 이 셀에서 에러가 나면(Colab의 CUDA 버전과 안 맞는 경우 등),
# 아래 줄 대신 CPU 버전으로 설치하세요: !pip install paddlepaddle paddleocr -q
!pip install paddlepaddle-gpu paddleocr -q

## 2. GPU 확인
출력이 `False`로 나오면 런타임 유형이 GPU로 설정되지 않은 것입니다. 상단 메뉴에서 런타임 > 런타임 유형 변경 > T4 GPU로 바꾸고 다시 실행하세요.

In [ ]:
import torch
print("GPU 사용 가능:", torch.cuda.is_available())

## 3. Google Drive 연동
실행하면 인증 창이 뜹니다. 로그인 후 권한을 허용하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. 경로 설정

**두 가지 방식 중 하나를 고르세요.**

- **A. 이미지가 이미 폴더로 풀려있는 경우**: 바로 그 폴더 경로를 `IMAGE_FOLDER`에 적으면 됩니다. (예: `/content/drive/MyDrive/novel_scans`)
- **B. zip 파일(안에 zip이 또 있는 중첩 구조 포함)인 경우**: `IMAGE_FOLDER`에 그 zip 파일 경로를 적으세요. (예: `/content/drive/MyDrive/novel_scans.zip`) 바로 다음 5번 셀이 자동으로 끝까지 풀어줍니다.

In [ ]:
IMAGE_FOLDER = '/content/drive/MyDrive/novel_scans'   # 스캔 이미지 폴더 또는 zip 파일 경로
OUTPUT_FOLDER = '/content/drive/MyDrive/novel_ocr_output'  # 결과 저장 폴더
TEXT_MIRROR_FOLDER = f'{OUTPUT_FOLDER}/texts'  # 원본과 같은 폴더 구조로 txt가 생기는 위치

import os
os.makedirs(TEXT_MIRROR_FOLDER, exist_ok=True)
print('이미지 폴더/파일:', IMAGE_FOLDER)
print('결과 폴더 (원본과 동일한 구조로 txt 생성):', TEXT_MIRROR_FOLDER)

## 5. (zip인 경우만) 중첩 압축 재귀적으로 풀기

`IMAGE_FOLDER`가 zip 파일이거나, 폴더 안에 폴더+zip이 뒤섞인 구조여도 상관없습니다.
아래 셀이 찾을 수 있는 모든 zip을 계속 풀어나가면서(zip 속의 zip까지) 끝까지 펼쳐줍니다.
다 풀린 이미지들은 `EXTRACT_FOLDER`에 원래 폴더 구조를 유지한 채로 모입니다.

이미지가 이미 폴더로 풀려있는 상태(A번)라면 이 셀은 그냥 실행해도 무방합니다 (풀 zip이 없으면 폴더만 복사됨).

In [ ]:
import zipfile
import shutil

EXTRACT_FOLDER = '/content/extracted_novel'  # 로컬(Colab 세션 디스크)에 풀 위치

# 매번 깨끗하게 다시 풀고 싶으면 아래 두 줄 주석 해제
# if os.path.exists(EXTRACT_FOLDER):
#     shutil.rmtree(EXTRACT_FOLDER)
os.makedirs(EXTRACT_FOLDER, exist_ok=True)

def extract_all_nested_zips(root_folder):
    """root_folder 안(하위 폴더 포함)의 모든 zip을 찾아서,
    각 zip을 자기 위치에 풀고, 풀린 결과 안에 또 zip이 있으면 계속 반복해서 푼다."""
    while True:
        zip_paths = []
        for dirpath, _, filenames in os.walk(root_folder):
            for fn in filenames:
                if fn.lower().endswith('.zip'):
                    zip_paths.append(os.path.join(dirpath, fn))
        if not zip_paths:
            break  # 더 이상 풀 zip이 없으면 종료
        for zp in zip_paths:
            extract_to = zp[:-4]  # 같은 이름의 폴더에 풀기 (파일명.zip -> 파일명/)
            os.makedirs(extract_to, exist_ok=True)
            try:
                with zipfile.ZipFile(zp, 'r') as zf:
                    zf.extractall(extract_to)
                print('풀림:', os.path.relpath(zp, root_folder))
            except zipfile.BadZipFile:
                print('손상되었거나 zip이 아님, 건너뜀:', zp)
            finally:
                os.remove(zp)  # 다 풀었으면 원본 zip은 삭제 (다음 순회에서 중복 감지 방지)

# 1) IMAGE_FOLDER 자체가 zip 파일 하나인 경우: 먼저 EXTRACT_FOLDER로 풀기
if os.path.isfile(IMAGE_FOLDER) and IMAGE_FOLDER.lower().endswith('.zip'):
    with zipfile.ZipFile(IMAGE_FOLDER, 'r') as zf:
        zf.extractall(EXTRACT_FOLDER)
    print('최초 zip 풀기 완료 ->', EXTRACT_FOLDER)
# 2) IMAGE_FOLDER가 폴더인 경우: 그 안의 파일들을 EXTRACT_FOLDER로 복사(이미 풀린 이미지 + zip 섞여있을 수 있음)
elif os.path.isdir(IMAGE_FOLDER):
    if os.path.abspath(IMAGE_FOLDER) != os.path.abspath(EXTRACT_FOLDER):
        shutil.copytree(IMAGE_FOLDER, EXTRACT_FOLDER, dirs_exist_ok=True)
    print('폴더 복사 완료 ->', EXTRACT_FOLDER)
else:
    raise FileNotFoundError(f'경로를 찾을 수 없습니다: {IMAGE_FOLDER}')

# 3) 그 안에 남아있는 모든 중첩 zip을 끝까지 재귀적으로 풀기
extract_all_nested_zips(EXTRACT_FOLDER)

# 4) 이후 셀들이 이 풀린 폴더를 보도록 IMAGE_FOLDER 갱신
IMAGE_FOLDER = EXTRACT_FOLDER
print('\n압축 해제 완료. 이제 IMAGE_FOLDER =', IMAGE_FOLDER)

## 6. OCR 엔진 준비 (PaddleOCR)
처음 실행 시 모델을 다운로드하기 때문에 1~2분 걸릴 수 있습니다.

In [ ]:
from paddleocr import PaddleOCR

# lang='korean': 한글 인쇄체 인식 모델
ocr_engine = PaddleOCR(use_angle_cls=True, lang='korean')

def run_ocr(img_path):
    """PaddleOCR 버전에 따라 API가 조금 다를 수 있어 둘 다 시도합니다."""
    try:
        # PaddleOCR 2.x 계열 API
        result = ocr_engine.ocr(img_path, cls=True)
        lines = []
        if result and result[0]:
            for line in result[0]:
                lines.append(line[1][0])
        return '\n'.join(lines)
    except AttributeError:
        # PaddleOCR 3.x 계열 API (predict 기반)
        result = ocr_engine.predict(img_path)
        lines = []
        for res in result:
            texts = res.get('rec_texts') or res.get('rec_text') or []
            if isinstance(texts, str):
                texts = [texts]
            lines.extend(texts)
        return '\n'.join(lines)

## 6b. (선택) 괄호 안 한자 보정

한글 모델(`lang='korean'`)의 사전에는 한자가 없어서, `진동왕(鎭東王)`처럼 괄호 안에 한자가 있으면
엉뚱한 한글로 잘못 인식됩니다. 아래 셀은 이걸 보정하는 실험적 기능입니다.

**동작 방식**: 한 줄을 인식한 뒤, 그 텍스트 안에서 `( )` 위치를 찾고 → 그 글자가 줄 전체에서 몇 번째
위치인지 비율로 계산해서 원본 이미지에서 그 부분만 잘라냄 → 한자 전용 모델(`chinese_cht`)로 그 조각만
다시 인식 → 괄호 안 내용을 그 결과로 교체합니다.

**전제조건(중요)**: 이 방식은 글자 간격이 고르게 정렬된(등간격) 인쇄체일 때 잘 맞습니다. 스캔본이
말씀하신 예시 이미지들처럼 양쪽 정렬/고정폭 느낌이면 잘 작동할 가능성이 높습니다. 100% 정확하진
않을 수 있어서, 몇 페이지 돌려보고 결과를 보여주시면 크롭 여백(`pad_ratio`) 등을 같이 조정해드릴게요.

**한자 대응을 끄고 싶으면** 바로 다음 셀에서 `HANDLE_HANJA = False`로 바꾸면 기존 한글 전용 방식으로 동작합니다.

In [ ]:
import re
import numpy as np
from PIL import Image

HANDLE_HANJA = True   # 괄호 안 한자 보정 켜기/끄기

# 한자(번체에 가까운 한국식 한자) 전용 모델. HANDLE_HANJA=False면 로드하지 않음
ocr_hanja = PaddleOCR(use_angle_cls=True, lang='chinese_cht') if HANDLE_HANJA else None

_PAREN_RE = re.compile(r'\(([^()]*)\)')


def _line_bbox(box):
    xs = [p[0] for p in box]
    ys = [p[1] for p in box]
    return min(xs), min(ys), max(xs), max(ys)


def _crop_char_range(image, box, n_chars, start_idx, end_idx, pad_ratio=0.15):
    """box(줄 전체 영역) 안에서 문자 인덱스 [start_idx:end_idx)에 해당하는 부분만
    비율로 계산해서 잘라낸다. 글자 간격이 고르다는 가정 하에 동작."""
    x1, y1, x2, y2 = _line_bbox(box)
    if n_chars <= 0:
        return None
    w = x2 - x1
    cx1 = x1 + w * (start_idx / n_chars)
    cx2 = x1 + w * (end_idx / n_chars)
    pad = max((cx2 - cx1) * pad_ratio, 2)
    cx1 = max(x1, cx1 - pad)
    cx2 = min(x2, cx2 + pad)
    if cx2 - cx1 < 2:
        return None
    return image.crop((int(cx1), int(y1), int(cx2), int(y2)))


def _recognize_crop(crop):
    if crop is None:
        return ''
    arr = np.array(crop.convert('RGB'))
    result = ocr_hanja.ocr(arr, cls=True)
    if not result or not result[0]:
        return ''
    return ''.join(t for _, (t, c) in result[0])


def run_ocr_mixed(img_path):
    """줄 단위로 한글 인식 후, 괄호 안 내용만 한자 모델로 재인식해서 교체."""
    result = ocr_engine.ocr(img_path, cls=True)
    lines = result[0] if result and result[0] else []
    # 위->아래 순서로 정렬
    lines.sort(key=lambda item: sum(p[1] for p in item[0]) / 4)

    pil_img = Image.open(img_path) if HANDLE_HANJA else None

    out_lines = []
    for box, (text, conf) in lines:
        if HANDLE_HANJA and '(' in text and ')' in text:
            chars = list(text)
            n = len(chars)
            # 뒤에서부터 교체해야 인덱스가 안 밀림
            matches = list(_PAREN_RE.finditer(text))
            for m in reversed(matches):
                inner_s, inner_e = m.start(1), m.end(1)
                if inner_e <= inner_s:
                    continue
                crop = _crop_char_range(pil_img, box, n, inner_s, inner_e)
                han_text = _recognize_crop(crop)
                if han_text:
                    chars[inner_s:inner_e] = list(han_text)
            text = ''.join(chars)
        out_lines.append(text)

    return '\n'.join(out_lines)

## 7. 전체 이미지 일괄 처리 (원본 폴더 구조 유지)

압축을 풀었다면 하위 폴더 구조가 남아있으므로, 하위 폴더까지 전부 재귀적으로 훑어서
이미지 파일을 모두 찾습니다. 이미지 하나마다 텍스트 파일 하나를 만들고, 그 텍스트 파일은
`TEXT_MIRROR_FOLDER` 아래에 **원본 이미지와 똑같은 하위 폴더 경로**로 저장됩니다.

예: `IMAGE_FOLDER/1장/01.jpg` → `TEXT_MIRROR_FOLDER/1장/01.txt`

이미 처리된 이미지는 같은 이름의 txt가 있으면 건너뛰므로, 세션이 끊겨도 이어서 처리할 수 있습니다.

In [ ]:
import glob

# 하위 폴더까지 전부 재귀적으로 검색 (중첩 zip을 풀면 폴더 구조가 깊어질 수 있으므로)
# gif는 정지 이미지면 정상 인식됨 (애니메이션 gif는 첫 프레임만 인식)
# 필요하면 확장자 추가/수정 (jpg, jpeg, png, gif, bmp, webp 등)
IMAGE_EXTENSIONS = ['*.jpg', '*.jpeg', '*.png', '*.gif', '*.bmp', '*.webp']

image_files = []
for ext in IMAGE_EXTENSIONS:
    image_files += glob.glob(os.path.join(IMAGE_FOLDER, '**', ext), recursive=True)
image_files = sorted(image_files)

print(f'총 {len(image_files)}개 이미지 발견')

done_count = 0
for i, img_path in enumerate(image_files):
    rel_path = os.path.relpath(img_path, IMAGE_FOLDER)          # 예: 1장/01.jpg
    rel_txt_path = os.path.splitext(rel_path)[0] + '.txt'        # 예: 1장/01.txt
    out_txt_path = os.path.join(TEXT_MIRROR_FOLDER, rel_txt_path)

    if os.path.exists(out_txt_path):
        done_count += 1
        continue  # 이미 처리된 이미지는 건너뜀 (재실행 시 이어서 처리)

    os.makedirs(os.path.dirname(out_txt_path), exist_ok=True)  # 원본과 같은 하위 폴더 생성

    text = run_ocr(img_path)

    with open(out_txt_path, 'w', encoding='utf-8') as f:
        f.write(text)

    print(f'[{i+1}/{len(image_files)}] 처리 완료: {rel_path} -> {rel_txt_path}')

print(f'\n모든 작업 완료! (이번 실행에서 건너뛴 기존 완료 파일: {done_count}개)')
print('결과 폴더:', TEXT_MIRROR_FOLDER)

## 8. 결과 확인

전체 txt 파일 개수와, 처음 몇 개 파일 내용을 미리 확인합니다.

In [ ]:
all_txt_files = sorted(glob.glob(os.path.join(TEXT_MIRROR_FOLDER, '**', '*.txt'), recursive=True))
print(f'생성된 txt 파일: {len(all_txt_files)}개\n')

for txt_path in all_txt_files[:3]:
    rel = os.path.relpath(txt_path, TEXT_MIRROR_FOLDER)
    with open(txt_path, 'r', encoding='utf-8') as f:
        preview = f.read()
    print(f'--- {rel} ---')
    print(preview[:300])
    print()